# Visualization of LLM Output Embeddings

Loads paragraph embeddings saved by `run_evaluation.py` and renders t-SNE scatter plots, colored by question type (positive / paraphrased positive / negative).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
import plotly.graph_objects as go

## 1 · Load Embeddings

In [ ]:
model_names = [
    "gpt-5.1", "claude_4.5_sonnet", "api-llama3.3", "api-llama4",
    "qwen3-4B", "qwen3-30B",
    "huatuo-7B", "huatuo-8B"
]

In [ ]:
EMBEDDINGS_FILE = "embeddings.npz"

data = np.load(EMBEDDINGS_FILE, allow_pickle=True)

pos_embeddings = data["pos_embeddings"]
neg_embeddings = data["neg_embeddings"]
neu_embeddings = data["neu_embeddings"]

pos_texts = data["pos_texts"].tolist()
neg_texts = data["neg_texts"].tolist()
neu_texts = data["neu_texts"].tolist()

print(f"Positive  : {pos_embeddings.shape}")
print(f"Negative  : {neg_embeddings.shape}")
print(f"Neutral   : {neu_embeddings.shape}")
print(f"Embedding dim: {pos_embeddings.shape[1]}")

## 2 · Combine & Label

In [ ]:
# Stack all embeddings into one matrix
all_embeddings = np.vstack([pos_embeddings, neg_embeddings, neu_embeddings])
all_texts      = pos_texts + neg_texts + neu_texts

# Numeric labels  0=positive  1=negative  2=neutral
labels = (
    [0] * len(pos_texts) +
    [1] * len(neg_texts) +
    [2] * len(neu_texts)
)
label_names = {0: "Positive", 1: "Negative", 2: "Neutral"}

# L2-normalise (optional but often improves t-SNE clustering)
all_embeddings_normed = normalize(all_embeddings, norm="l2")

print(f"Total samples: {len(all_texts)}  |  Embedding matrix: {all_embeddings.shape}")

## 3 · PCA Pre-reduction

Reduces dimensionality before t-SNE — speeds up computation and removes low-variance noise.
Standard recommendation is 50 components; we also show the adaptive approach (keep 95% variance).

In [ ]:
# ── PCA hyper-parameters ──────────────────────────────────────────────────────
# PCA_COMPONENTS : int  → fixed number of components (standard: 50)
#                  float 0–1 → keep enough components to explain that fraction of variance
# Set to min(50, n_features) so it works even on small/low-dim embeddings.
# ─────────────────────────────────────────────────────────────────────────────

n_features = all_embeddings_normed.shape[1]
PCA_COMPONENTS = min(50, n_features)   # ← change to e.g. 0.95 for adaptive mode

pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
embeddings_pca = pca.fit_transform(all_embeddings_normed)

var_explained = pca.explained_variance_ratio_.sum()
print(f"PCA: {all_embeddings_normed.shape} → {embeddings_pca.shape}")
print(f"Variance explained by {embeddings_pca.shape[1]} components: {var_explained:.1%}")

# ── Optional: scree plot to inspect explained variance ────────────────────────
fig_scree, ax_scree = plt.subplots(figsize=(7, 3))
fig_scree.patch.set_facecolor("#0f0f0f")
ax_scree.set_facecolor("#1a1a1a")
ax_scree.plot(
    range(1, len(pca.explained_variance_ratio_) + 1),
    pca.explained_variance_ratio_.cumsum(),
    color="#4CAF50", linewidth=2,
)
ax_scree.axhline(0.95, color="#F44336", linestyle="--", linewidth=1, label="95% threshold")
ax_scree.set_xlabel("Number of PCA components", color="#888")
ax_scree.set_ylabel("Cumulative variance explained", color="#888")
ax_scree.set_title("PCA Scree Plot", color="white")
ax_scree.tick_params(colors="#555")
for spine in ax_scree.spines.values(): spine.set_edgecolor("#333")
ax_scree.legend(labelcolor="white", facecolor="#222", edgecolor="#444")
plt.tight_layout()
plt.show()

## 4 · Run t-SNE (on PCA-reduced embeddings)

In [ ]:
# ── t-SNE hyper-parameters ────────────────────────────────────────────────────
# perplexity : balances local vs global structure; try 5–50 (rule of thumb: n_samples/3)
# n_iter     : more iterations → more stable layout (300–1000 is usually fine)
# metric     : 'euclidean' is standard after PCA (cosine distances are already
#              captured in the PCA space); switch to 'cosine' if skipping PCA
# init       : 'pca' gives a deterministic, often better starting layout
# ─────────────────────────────────────────────────────────────────────────────

N = len(all_texts)
perplexity = min(30, N - 1)   # 30 is the scikit-learn default; scales down for small datasets

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    n_iter=1000,
    metric="euclidean",   # euclidean is correct after PCA pre-reduction
    init="pca",
    random_state=42,
    verbose=1,
)

print(f"Running t-SNE on {N} samples with {embeddings_pca.shape[1]}-dim PCA input (perplexity={perplexity}) …")
coords_2d = tsne.fit_transform(embeddings_pca)   # ← PCA output, not raw embeddings
print("Done.")

## 5 · Static Matplotlib Plot

In [ ]:
PALETTE = {0: "#4CAF50", 1: "#F44336", 2: "#2196F3"}   # green / red / blue
MARKERS = {0: "o", 1: "s", 2: "^"}                       # circle / square / triangle

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor("#0f0f0f")
ax.set_facecolor("#1a1a1a")

labels_arr = np.array(labels)

for label_id, label_name in label_names.items():
    mask = labels_arr == label_id
    ax.scatter(
        coords_2d[mask, 0],
        coords_2d[mask, 1],
        c=PALETTE[label_id],
        marker=MARKERS[label_id],
        s=120,
        alpha=0.88,
        edgecolors="white",
        linewidths=0.4,
        label=label_name,
        zorder=3,
    )

# Annotate each point with a short excerpt
for i, (x, y, txt) in enumerate(zip(coords_2d[:, 0], coords_2d[:, 1], all_texts)):
    short = txt[:40] + "…" if len(txt) > 40 else txt
    ax.annotate(
        short,
        (x, y),
        fontsize=6.5,
        color="#dddddd",
        xytext=(5, 5),
        textcoords="offset points",
        path_effects=[pe.withStroke(linewidth=1.5, foreground="#1a1a1a")],
    )

ax.set_title("t-SNE – Question Paragraph Embeddings", color="white", fontsize=14, pad=14)
ax.set_xlabel("t-SNE dim 1", color="#888")
ax.set_ylabel("t-SNE dim 2", color="#888")
ax.tick_params(colors="#555")
for spine in ax.spines.values():
    spine.set_edgecolor("#333")

legend = ax.legend(framealpha=0.2, labelcolor="white", facecolor="#222", edgecolor="#444")

plt.tight_layout()
plt.savefig("tsne_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tsne_plot.png")

## 6 · Interactive Plotly Plot (hover to read full text)

In [ ]:
PLOTLY_COLORS = {0: "#4CAF50", 1: "#F44336", 2: "#2196F3"}
PLOTLY_SYMBOLS = {0: "circle", 1: "square", 2: "triangle-up"}

traces = []
for label_id, label_name in label_names.items():
    mask = labels_arr == label_id
    texts_subset = [all_texts[i] for i in range(len(all_texts)) if labels[i] == label_id]
    # Wrap long texts for readable tooltips
    hover_texts = ["<br>".join([t[i:i+60] for i in range(0, len(t), 60)]) for t in texts_subset]

    traces.append(go.Scatter(
        x=coords_2d[mask, 0],
        y=coords_2d[mask, 1],
        mode="markers",
        name=label_name,
        marker=dict(
            size=12,
            color=PLOTLY_COLORS[label_id],
            symbol=PLOTLY_SYMBOLS[label_id],
            opacity=0.85,
            line=dict(width=0.8, color="white"),
        ),
        text=hover_texts,
        hovertemplate="<b>%{text}</b><extra></extra>",
    ))

fig = go.Figure(
    data=traces,
    layout=go.Layout(
        title="t-SNE – Question Paragraph Embeddings (hover for full text)",
        xaxis_title="t-SNE dim 1",
        yaxis_title="t-SNE dim 2",
        plot_bgcolor="#1a1a1a",
        paper_bgcolor="#0f0f0f",
        font=dict(color="white"),
        legend=dict(bgcolor="#222", bordercolor="#444"),
        width=900,
        height=620,
    ),
)

fig.write_html("tsne_interactive.html")
fig.show()
print("Saved: tsne_interactive.html")

## 7 · Cosine-Similarity Heatmap (bonus)

Shows pairwise similarity between every paragraph — useful for sanity-checking that positive/negative clusters are really distinct.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(all_embeddings_normed)

short_labels = [t[:30] + "…" if len(t) > 30 else t for t in all_texts]

fig2, ax2 = plt.subplots(figsize=(12, 10))
fig2.patch.set_facecolor("#0f0f0f")
ax2.set_facecolor("#1a1a1a")

im = ax2.imshow(sim_matrix, cmap="RdYlGn", vmin=-0.2, vmax=1.0, aspect="auto")
plt.colorbar(im, ax=ax2, label="Cosine similarity")

ax2.set_xticks(range(N))
ax2.set_yticks(range(N))
ax2.set_xticklabels(short_labels, rotation=75, ha="right", fontsize=7, color="#ccc")
ax2.set_yticklabels(short_labels, fontsize=7, color="#ccc")
ax2.set_title("Pairwise Cosine Similarity Heatmap", color="white", fontsize=13, pad=12)

# Draw separator lines between categories
n_pos = len(pos_texts)
n_neg = len(neg_texts)
for offset in [n_pos - 0.5, n_pos + n_neg - 0.5]:
    ax2.axhline(offset, color="white", linewidth=1.2, alpha=0.5)
    ax2.axvline(offset, color="white", linewidth=1.2, alpha=0.5)

plt.tight_layout()
plt.savefig("similarity_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: similarity_heatmap.png")

# Matrix of Embeddings

- cosine of positive question answers across models
- cosine of negative question answers across models
